In [1]:

import numpy as np
import matplotlib.pyplot as plt
import time
from itertools import combinations

import jax
import jax.numpy as jnp

import importlib, os
import east
importlib.reload(east)

plt.rcParams['figure.dpi'] = 120



## Preliminary local-cluster DMD smoke test

This notebook is the first check of the new pooled per-site representation from `AGENTS.md`. Each equilibrium initial configuration supplies `N` local trajectories, one attached to each site, and the DMD fit pools snapshot pairs over both sites and configurations.

The dictionary is a downward-closed local cluster basis on the facilitating/right window `{i, ..., i+J}`. Product observables are evaluated on each walker first, averaged over walkers second, then analytically centered and scaled by the Bernoulli equilibrium value. This keeps connected correlations in the data.


In [2]:

key = jax.random.PRNGKey(11)

# Temperature list: change this single list to control which expensive trajectories are run.
ls = [4]

# Expensive trajectory-generation knobs.
N = 1024
n_configs = 16
n_walkers = 1024
n_records = 500

default_dt_record = 0.25
use_acf_window = True
acf_points_per_tau = 250.0

# Cheap dictionary/DMD knobs. Trajectories are generated once per l and reused for this sweep.
J_values = [0, 1, 2, 3, 4, 6]
p_order = 2
contiguity_mode = 'floating-contiguous'  # 'anchored-contiguous', 'floating-contiguous', or 'all-subsets'
svd_energy = 0.99
r_max = 80
eps_threshold = 5e-8

print(f'ls={ls}')
print(f'N={N}, configs={n_configs}, walkers/config={n_walkers}, records={n_records}')
print(f'J_values={J_values}, p={p_order}, mode={contiguity_mode}')


ls=[4]
N=1024, configs=16, walkers/config=1024, records=500
J_values=[0, 1, 2, 3, 4, 6], p=2, mode=floating-contiguous


In [ ]:

def sample_fixed_concentration_config(key, N, c_eq):
    """Equilibrium-like initial state with exactly round(N c_eq) excitations."""
    n_up = int(round(N * c_eq))
    return jax.random.permutation(
        key,
        jnp.concatenate([
            jnp.ones(n_up, dtype=jnp.int32),
            jnp.zeros(N - n_up, dtype=jnp.int32),
        ])
    )


def dt_for_length(l):
    if not use_acf_window or not os.path.exists('acf_results.npy'):
        return default_dt_record, None

    acf_results = np.load('acf_results.npy', allow_pickle=True).item()
    if l not in acf_results or 'tau' not in acf_results[l]:
        return default_dt_record, None

    tau_acf = float(acf_results[l]['tau'])
    return tau_acf / acf_points_per_tau, tau_acf


trajectory_data = {}
t0 = time.time()

for l in ls:
    T = float(east.temperature_from_length(l))
    c_eq = float(east.concentration(T))
    dt_record, tau_acf_ref = dt_for_length(l)
    times_l = np.arange(n_records + 1, dtype=np.float64) * dt_record

    all_config_trajs = []
    state_inits = []
    print(f'\nl={l}, T={T:.5g}, c_eq={c_eq:.5g}, dt={dt_record:.5g}, T_window={times_l[-1]:.5g}')
    if tau_acf_ref is not None:
        print(f'  tau_ACF reference={tau_acf_ref:.5g}; window={times_l[-1] / tau_acf_ref:.2f} tau_ACF')

    for config_idx in range(n_configs):
        key, init_key, run_key = jax.random.split(key, 3)
        state_init = sample_fixed_concentration_config(init_key, N, c_eq)
        mean_traj, walker_trajs = east.isoconfigurational_ensemble(
            N, T, dt_record, n_records, n_walkers, state_init, run_key
        )
        walker_np = np.asarray(walker_trajs, dtype=np.int8)
        all_config_trajs.append(walker_np)
        state_inits.append(np.asarray(state_init, dtype=np.int8))
        print(f'  config {config_idx + 1:2d}/{n_configs}: walkers {walker_np.shape}')

    trajectory_data[l] = {
        'T': T,
        'c_eq': c_eq,
        'dt_record': dt_record,
        'tau_acf_ref': tau_acf_ref,
        'times': times_l,
        'walker_trajs': np.stack(all_config_trajs, axis=0),  # (configs, walkers, time, N)
        'state_inits': np.stack(state_inits, axis=0),        # (configs, N)
    }
    print(f'  stored walker_trajs {trajectory_data[l]["walker_trajs"].shape}')

print(f'\ntrajectory generation finished in {time.time() - t0:.1f}s')



l=4, T=0.91024, c_eq=0.25, dt=0.17975, T_window=89.877
  tau_ACF reference=44.938; window=2.00 tau_ACF
  config  1/16: walkers (1024, 501, 1024)
  config  2/16: walkers (1024, 501, 1024)
  config  3/16: walkers (1024, 501, 1024)



## Local Cluster Dictionary

The cluster list contains the constant observable plus monomials in the one-sided facilitating window. For every non-constant cluster, the product is computed per walker and only then averaged over walkers. Standardization uses `<chi_S>_eq = c_eq ** |S|` and `Var(chi_S)_eq = c_eq ** |S| * (1 - c_eq ** |S|)`.


In [ ]:

def local_clusters(J, p, mode='floating-contiguous'):
    """Return downward-closed offset tuples inside {0, ..., J}."""
    offsets = list(range(J + 1))
    clusters = []

    if mode == 'anchored-contiguous':
        max_len = min(p, J + 1)
        clusters = [tuple(range(length)) for length in range(1, max_len + 1)]
    elif mode == 'floating-contiguous':
        for length in range(1, min(p, J + 1) + 1):
            for start in range(0, J - length + 2):
                clusters.append(tuple(range(start, start + length)))
    elif mode == 'all-subsets':
        for order in range(1, min(p, J + 1) + 1):
            clusters.extend(tuple(c) for c in combinations(offsets, order))
    else:
        raise ValueError(f'unknown contiguity mode: {mode}')

    return [()] + clusters


def walker_cluster_average(walker_trajs, clusters):
    """
    Evaluate local products per walker, then average over walkers.

    walker_trajs: (walkers, time, N) binary array.
    returns: (time, N, D) with D=len(clusters).
    """
    walker_trajs = np.asarray(walker_trajs, dtype=np.float64)
    W, n_time, n_sites = walker_trajs.shape
    features = np.empty((n_time, n_sites, len(clusters)), dtype=np.float64)
    features[:, :, 0] = 1.0

    for feature_idx, cluster in enumerate(clusters[1:], start=1):
        product = np.ones_like(walker_trajs, dtype=np.float64)
        for offset in cluster:
            product *= np.roll(walker_trajs, -offset, axis=2)
        features[:, :, feature_idx] = product.mean(axis=0)

    return features


def standardize_cluster_features(cluster_means, clusters, c_eq):
    standardized = np.array(cluster_means, dtype=np.float64, copy=True)
    for feature_idx, cluster in enumerate(clusters[1:], start=1):
        order = len(cluster)
        mean_eq = c_eq ** order
        var_eq = mean_eq * (1.0 - mean_eq)
        standardized[:, :, feature_idx] = (standardized[:, :, feature_idx] - mean_eq) / np.sqrt(var_eq)
    return standardized


def build_local_psis(walker_trajs_by_config, clusters, c_eq):
    psis = []
    raw_means = []
    for walker_trajs in walker_trajs_by_config:
        cluster_means = walker_cluster_average(walker_trajs, clusters)
        raw_means.append(cluster_means)
        psis.append(standardize_cluster_features(cluster_means, clusters, c_eq))
    return np.stack(psis, axis=0), np.stack(raw_means, axis=0)


def snapshot_pairs_from_psis(Psis):
    """Pool snapshot pairs over configurations and sites."""
    X = Psis[:, :-1, :, :].reshape(-1, Psis.shape[-1]).T
    Y = Psis[:, 1:, :, :].reshape(-1, Psis.shape[-1]).T
    return X, Y


for J in J_values:
    clusters = local_clusters(J, p_order, contiguity_mode)
    print(f'J={J}: D={len(clusters)} clusters; first clusters={clusters[:min(8, len(clusters))]}')


In [ ]:

def fit_exact_dmd(X, Y, dt, energy=0.99, r_max=None):
    """Exact DMD with energy-based SVD truncation."""
    U, s, Vh = np.linalg.svd(X, full_matrices=False)
    cumulative = np.cumsum(s**2) / np.sum(s**2)
    r = int(np.searchsorted(cumulative, energy) + 1)
    if r_max is not None:
        r = min(r, int(r_max))

    Ur = U[:, :r]
    sr = s[:r]
    Vr = Vh.conj().T[:, :r]

    A_tilde = Ur.conj().T @ Y @ Vr @ np.diag(1.0 / sr)
    lam, W = np.linalg.eig(A_tilde)
    Phi = Y @ Vr @ np.diag(1.0 / sr) @ W
    mu = np.log(lam.astype(complex)) / dt

    residuals = np.empty_like(mu, dtype=np.float64)
    for i in range(len(mu)):
        projected_before = Ur @ W[:, i]
        projected_after = Phi[:, i]
        denom = np.linalg.norm(projected_before)
        residuals[i] = np.linalg.norm(projected_after - lam[i] * projected_before) / max(denom, 1e-15)

    return {
        'U': Ur,
        's': sr,
        'rank': r,
        'lambda': lam,
        'mu': mu,
        'Phi': Phi,
        'A_tilde': A_tilde,
        'residuals': residuals,
    }


def reconstruct_dmd(fit, psi0, times):
    """Reconstruct feature trajectory from one initial local dictionary vector."""
    Phi = fit['Phi']
    mu = fit['mu']
    b = np.linalg.lstsq(Phi, psi0, rcond=None)[0]
    dynamics = np.exp(np.outer(mu, times)) * b[:, None]
    return (Phi @ dynamics).T


def reconstruct_initial_samples(fit, psi0_samples, times, chunk_size=4096):
    pieces = []
    for start in range(0, psi0_samples.shape[0], chunk_size):
        stop = min(start + chunk_size, psi0_samples.shape[0])
        pieces.append(np.stack([
            reconstruct_dmd(fit, psi0, times)
            for psi0 in psi0_samples[start:stop]
        ], axis=0))
    return np.concatenate(pieces, axis=0)


def reconstruction_error(Psis, Psi_hats):
    """Average relative L2 error over standardized non-constant local trajectories."""
    target = Psis[..., 1:]
    pred = np.real(Psi_hats[..., 1:])
    numerator = np.linalg.norm(pred - target)
    denominator = np.linalg.norm(target)
    return numerator / denominator


def mode_summary(fit, times, max_rows=6):
    mu = fit['mu']
    residuals = fit['residuals']
    decaying = np.real(mu) < -1e-12
    tau = np.full(mu.shape, np.inf, dtype=float)
    tau[decaying] = 1.0 / np.abs(np.real(mu[decaying]))
    good = residuals < eps_threshold
    slow = np.where(decaying & good)[0]
    slow = sorted(slow, key=lambda i: tau[i], reverse=True)[:max_rows]
    rows = []
    for i in slow:
        rows.append((i, mu[i].real, mu[i].imag, tau[i], residuals[i]))
    return rows, good



## Phase-0 Style J Sweep

This is intentionally modest: trajectories are generated only for `ls = [3, 4, 5, 6]`, then exact DMD is fit for each `(l, J)` using the chosen `p_order` and contiguity mode. The main quick diagnostic is pooled local reconstruction error versus `J`.


In [ ]:

sweep_results = {}

for l, data in trajectory_data.items():
    sweep_results[l] = {}
    print(f'\n=== l={l} ===')
    for J in J_values:
        t0 = time.time()
        clusters = local_clusters(J, p_order, contiguity_mode)
        Psis, raw_means = build_local_psis(data['walker_trajs'], clusters, data['c_eq'])
        X, Y = snapshot_pairs_from_psis(Psis)
        fit = fit_exact_dmd(X, Y, data['dt_record'], energy=svd_energy, r_max=r_max)

        psi0_samples = Psis[:, 0, :, :].reshape(-1, Psis.shape[-1])
        Psi_hat_samples = reconstruct_initial_samples(fit, psi0_samples, data['times'])
        Psi_hats = Psi_hat_samples.reshape(n_configs, N, len(data['times']), len(clusters)).transpose(0, 2, 1, 3)
        rel_err = reconstruction_error(Psis, Psi_hats)
        slow_rows, good_mask = mode_summary(fit, data['times'])

        sweep_results[l][J] = {
            'clusters': clusters,
            'Psis': Psis,
            'raw_means': raw_means,
            'fit': fit,
            'Psi_hats': Psi_hats,
            'rel_err': rel_err,
            'n_resdmd_good': int(np.sum(good_mask)),
            'elapsed': time.time() - t0,
        }

        print(
            f'J={J:2d}, D={len(clusters):3d}, rank={fit["rank"]:3d}, '
            f'good_res={np.sum(good_mask):3d}, rel_err={rel_err:.3e}, '
            f'time={sweep_results[l][J]["elapsed"]:.1f}s'
        )
        if slow_rows:
            print('  slow low-residual decays: idx  Re(mu)      Im(mu)      tau        residual')
            for row in slow_rows[:3]:
                print(f'  {row[0]:3d}  {row[1]: .3e}  {row[2]:+.3e}  {row[3]: .3e}  {row[4]:.1e}')
        else:
            print('  no decaying modes below eps_threshold')


In [ ]:

fig, ax = plt.subplots(figsize=(6.4, 4.0))
for l in ls:
    errs = [sweep_results[l][J]['rel_err'] for J in J_values]
    ax.plot(J_values, errs, marker='o', lw=2.0, label=f'l={l}')

ax.set_yscale('log')
ax.set_xlabel('local window reach J')
ax.set_ylabel('pooled relative L2 reconstruction error')
ax.set_title(f'Local-cluster DMD J sweep, p={p_order}, {contiguity_mode}')
ax.legend(frameon=False)
ax.grid(True, which='both', alpha=0.18)
fig.tight_layout()


In [ ]:

# Choose a representative fit for the detailed plots below.
plot_l = ls[0]
plot_J = J_values[-1]
plot_config = 0
plot_site = 0

plot_data = trajectory_data[plot_l]
plot_result = sweep_results[plot_l][plot_J]
clusters = plot_result['clusters']
Psis = plot_result['Psis']
Psi_hats = plot_result['Psi_hats']
times = plot_data['times']
c_eq = plot_data['c_eq']

print(f'plot_l={plot_l}, plot_J={plot_J}, D={len(clusters)}, rel_err={plot_result["rel_err"]:.3e}')
print(f'clusters={clusters}')


In [ ]:

fig, ax = plt.subplots(figsize=(6.6, 4.1))
marker_stride = max(1, len(times) // 28)

for feature_idx, cluster in enumerate(clusters[1:], start=1):
    measured = Psis[plot_config, :, plot_site, feature_idx]
    predicted = np.real(Psi_hats[plot_config, :, plot_site, feature_idx])
    label = 's_i' if cluster == (0,) else str(cluster)
    ax.plot(times, measured, lw=1.8, alpha=0.75, label=f'{label} measured')
    ax.plot(times, predicted, marker='o', ls='None', ms=3.2, mfc='white',
            mew=0.9, markevery=marker_stride, alpha=0.8, label=f'{label} DMD')
    if feature_idx >= min(5, len(clusters) - 1):
        break

ax.axhline(0.0, color='0.35', ls=':', lw=1.2)
ax.set_title(f'Standardized local observables, l={plot_l}, J={plot_J}, site={plot_site}')
ax.set_xlabel('time')
ax.set_ylabel('standardized observable')
ax.margins(x=0)
ax.legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()


In [ ]:

# Site-propensity reconstruction from the cluster dictionary: feature (0,) is standardized s_i.
site_feature_idx = clusters.index((0,))
sigma_site = np.sqrt(c_eq * (1.0 - c_eq))
site_measured = Psis[plot_config, :, :, site_feature_idx] * sigma_site + c_eq
site_predicted = np.real(Psi_hats[plot_config, :, :, site_feature_idx]) * sigma_site + c_eq
state_init = plot_data['state_inits'][plot_config]
up_mask = state_init == 1
dn_mask = state_init == 0

fig, ax = plt.subplots(figsize=(6.6, 4.1))
ax.plot(times, site_measured[:, up_mask].mean(axis=1), color='tab:red', lw=2.4,
        label='initially up, walker product mean')
ax.plot(times, site_measured[:, dn_mask].mean(axis=1), color='tab:blue', lw=2.4,
        label='initially down, walker product mean')
ax.plot(times, site_predicted[:, up_mask].mean(axis=1), color='tab:red', marker='o',
        ls='None', ms=4.0, mfc='white', mec='tab:red', mew=1.0, markevery=marker_stride,
        label='initially up, DMD')
ax.plot(times, site_predicted[:, dn_mask].mean(axis=1), color='tab:blue', marker='s',
        ls='None', ms=3.8, mfc='white', mec='tab:blue', mew=1.0, markevery=marker_stride,
        label='initially down, DMD')
ax.axhline(c_eq, color='0.35', ls=':', lw=1.2, label='c_eq')

ax.set_title(f'Site-propensity reconstruction, l={plot_l}, J={plot_J}')
ax.set_xlabel('time')
ax.set_ylabel('site propensity')
ax.margins(x=0)
ax.legend(frameon=False, ncol=2, fontsize=8)
fig.tight_layout()


In [ ]:

def dmd_pseudospectrum_mu(fit, dt, n_real=80, n_imag=60, real_pad=0.08, imag_pad=0.12):
    """Finite-dimensional DMD pseudospectrum in the continuous-time mu plane."""
    A = fit['A_tilde']
    mu = fit['mu']
    real = np.real(mu[np.isfinite(mu)])
    imag = np.imag(mu[np.isfinite(mu)])

    re_span = max(real.max() - real.min(), 1e-6)
    im_span = max(imag.max() - imag.min(), 1e-6)
    re_grid = np.linspace(real.min() - real_pad * re_span,
                          max(0.0, real.max()) + real_pad * re_span,
                          n_real)
    im_grid = np.linspace(imag.min() - imag_pad * im_span,
                          imag.max() + imag_pad * im_span,
                          n_imag)

    eye = np.eye(A.shape[0], dtype=complex)
    sigma_min = np.empty((n_imag, n_real), dtype=float)
    for j, im in enumerate(im_grid):
        for i, re in enumerate(re_grid):
            z = np.exp((re + 1j * im) * dt)
            sigma_min[j, i] = np.linalg.svd(z * eye - A, compute_uv=False)[-1]

    return re_grid, im_grid, np.log10(np.maximum(sigma_min, 1e-16))


plot_pseudospectrum = True
fit = plot_result['fit']
mu = fit['mu']
residuals = fit['residuals']
low_res = residuals < eps_threshold

fig, ax = plt.subplots(figsize=(5.8, 4.4))

if plot_pseudospectrum:
    t0 = time.time()
    re_grid, im_grid, log_sigma_min = dmd_pseudospectrum_mu(fit, plot_data['dt_record'])
    levels = np.linspace(np.nanpercentile(log_sigma_min, 5),
                         np.nanpercentile(log_sigma_min, 95), 18)
    cf = ax.contourf(re_grid, im_grid, log_sigma_min, levels=levels,
                     cmap='magma_r', alpha=0.82)
    cbar = fig.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label('log10 pseudospectrum residual')
    print(f'pseudospectrum grid computed in {time.time() - t0:.1f}s')

ax.axvline(0, color='0.7', lw=1)
ax.axhline(0, color='0.7', lw=1)
ax.plot(np.real(mu[~low_res]), np.imag(mu[~low_res]), 'o', ms=4.2,
        color='white', mec='0.45', mew=0.7, alpha=0.85, label='filtered out')
ax.plot(np.real(mu[low_res]), np.imag(mu[low_res]), 'o', ms=5.0,
        color='tab:green', mec='black', mew=0.6, alpha=0.95,
        label=f'residual < {eps_threshold:g}')
ax.set_xlabel('Re(mu)')
ax.set_ylabel('Im(mu)')
ax.set_title(f'Local-cluster DMD spectrum, l={plot_l}, J={plot_J}')
ax.legend(frameon=False, loc='best')
fig.tight_layout()



## Notes

This notebook is still a preliminary gate check. Use the `J_values`, `p_order`, and `contiguity_mode` cells above to probe whether reconstruction quality plateaus near the expected East length scale before treating extracted timescales as meaningful.
